# Course notebook — Agent Systems

Design agentic systems as bounded stateful architectures rather than treating autonomy as a model feature.

Use Python 3.10 or later. All inputs are included and the workbook runs without API keys. Each lecture demonstration is a small model of the lecture mechanism. Run it, compare the result with your prediction, and use the section checkpoint to record what you learned.


## Practice case · Tuesday's volunteer calendar

Lena coordinates a volunteer program. Choosing a candidate schedule can be flexible, but publishing repeated or unapproved calendar changes wastes other people’s time. She needs the workflow to survive overnight outages with a comprehensible activity history.

The workplace scenario is fictional.


## Workflows and agent loops


### Lecture 01 · Agent or Workflow?

Define agents by goal-directed action selection in a loop and contrast them with deterministic workflows and single tool calls.

#### Decide which step needs discretion

The notice period and shift duration are fixed. Finding candidates is ambiguous because volunteers describe availability in natural language.

Use a fixed workflow for constraints and approval. Bound a model-assisted step to interpreting availability and suggesting candidates. Test each transition without requiring a model call.

#### Read the mechanism

**agency**

Agency is goal-directed action selection in a loop—observe, decide, act—rather than a single model call. It implies autonomy bounded by policy, tools, and termination rules.

An agent chooses which tool to call next based on observations, unlike a fixed workflow script.

**workflows**

Workflows are deterministic orchestrations with predefined steps, branches, and error handlers. They excel when paths are known and compliance requires repeatability.

Invoice approval always follows submit → manager → finance with explicit gates.

**state machines**

State machines model allowed statuses and transitions explicitly, making illegal steps unrepresentable. They clarify where agents pause, resume, or terminate.

Ticket automation states: open → pending_approval → resolved with defined transition triggers.

**autonomy**

Autonomy is how much discretion the system has to choose actions without human approval. More autonomy demands stronger evals, budgets, and rollback.

Auto-closing duplicate tickets is low autonomy; auto-issuing refunds is high and needs gates.

**control**

Control mechanisms—approvals, rate limits, tool allowlists— constrain agent behavior within safe envelopes. Control is designed, not emergent from prompts alone.

Payments above $500 require human approval even if the agent recommends proceed.

#### Worked example

**Situation:** A multi-step task may pause for hours and must resume without repeating side effects. Product wants an "agent" for employee onboarding; ops wants predictable workflows.

**Baseline:** Hard-coded workflow with 12 steps—breaks when vendor API response order changes.

**Application:** Model same onboarding task as deterministic workflow vs goal-directed agent loop; compare failure handling when optional branch appears; document where autonomy earns its cost.

**Test cases:** (1) Normal: happy-path hire with all docs present. (2) Boundary: optional visa check branch. (3) Adversarial: external API returns transient 503 mid-flow.

**Measurement:** Completion rate, recovery steps, human interventions per path; side-effect duplication count (must be zero).

**Design question:** Which onboarding subtask justifies agent autonomy over a workflow state machine?

**Predict before running.** Use the least autonomy that handles the uncertainty in the task. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 8.1: Agent or Workflow?"""

WORKFLOW = ["create_account", "assign_laptop", "grant_access"]
AGENT = {"goal": "complete onboarding", "actions": WORKFLOW, "replans": True}
def run(steps, fail_at=None):
    done = []
    for i, s in enumerate(steps):
        if fail_at == i:
            return done, "paused"
        done.append(s)
    return done, "complete"
print("workflow:", run(WORKFLOW, fail_at=1))
print("agent can resume:", AGENT["replans"])


#### Interpret and apply

Model the same task as a workflow and as an agent, then compare.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. How do agents differ from workflows in handling unexpected observations?
2. When is autonomy unnecessary cost?
3. What baseline uses fixed scripts only?

<details><summary>Check your reasoning</summary>

Q1: Agents replan; workflows need predefined branches. Q2: Fully deterministic tasks with stable APIs. Q3: Static DAG with no replanning.

</details>


### Lecture 02 · The Agent Loop

Connect goal, state, planning, action, observation, reflection, and termination into a bounded state machine.

#### A loop that stops early is often the right result

The scheduler has exhausted its candidate slots and considers asking a calendar tool again despite unchanged inputs.

Make progress observable and set a stopping condition tied to budget and evidence. A stop with an explanation and a next action gives Lena something to work with. Repetition without new state gives her a delay.

#### Read the mechanism

**plan-act-observe**

Plan–act–observe separates choosing the next action, executing it, and recording observations that update state.

Agent plans 'create draft', executes, observes 'draft id=7', then plans verification instead of repeating creation.

**state**

State captures variables the system believes true at a point in execution—inventory, user intent, pending approvals. Explicit state enables recovery and verification.

Agent state tracks current_step, artifacts_created, and budget_remaining across turns.

**reflection**

Reflection lets agents critique recent actions and adjust strategy—retry, replan, or escalate. Without reflection, loops repeat the same failing action.

After tool 403, reflect and switch to read-only search instead of retrying delete.

**termination**

Termination criteria stop search, agent loops, or generation when goals are met, budgets exhausted, or progress stalls. Without them, systems loop indefinitely.

Stop after five tool calls, success, or three consecutive no-progress iterations.

**budgets**

Budgets cap tokens, tool calls, wall time, or dollars per task or session. Hard budgets prevent runaway agents and make economics predictable.

A research agent stops after $0.50 API spend or ten tool calls, whichever comes first.

#### Worked example

**Situation:** The onboarding agent runs plan-act-observe cycles but previously spiraled on repeated failed API calls.

**Baseline:** while True loop calling model until success—no termination budget.

**Application:** Implement bounded state machine: goal, step counter, max attempts, reflection on failure, explicit termination states; log observations each cycle.

**Test cases:** (1) Normal: three-step plan completes. (2) Boundary: hits max attempts exactly. (3) Adversarial: tool returns success but wrong employee ID.

**Measurement:** Steps to completion, loop termination compliance, false-success detection rate.

**Design question:** Which state variable prevents an unreliable retry loop masquerading as an agent?

**Predict before running.** An agent loop without explicit state and stopping rules is an unreliable retry loop. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 8.2: The Agent Loop"""

CHAPTER = "8.2"
print("chapter hook:", CHAPTER)
state = {"step": 0, "observations": [], "done": False, "budget": 3}
while not state["done"] and state["step"] < state["budget"]:
    state["step"] += 1
    obs = f"obs-{state['step']}"
    state["observations"].append(obs)
    state["done"] = state["step"] == 3
print(state)
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Extend the included agent loop with failures and checkpointing.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why must agent loops have explicit termination rules?
2. How does reflection differ from blind retry?
3. What baseline lacks state tracking?

<details><summary>Check your reasoning</summary>

Q1: Unbounded loops burn cost and duplicate effects. Q2: Reflection diagnoses failure cause before next action. Q3: Retry-until-success with no budget.

</details>


**Section checkpoint.** Identify where a fixed workflow is sufficient. Write an invariant for a bounded loop.


## Recovery and patterns


### Lecture 03 · Agent Memory and Recovery

Manage working state, episodic history, durable checkpoints, resumability, compensation, and idempotent tools.

#### The service is back after midnight

An approval arrived before the worker restarted. The schedule proposal and its approval must stay paired after recovery.

Persist the reviewed proposal, the decision, and the next valid transition. Exercise restart separately from replay so the team can check both continuity and duplicate prevention.

#### Read the mechanism

**checkpoints**

Checkpoints persist durable agent state so interrupted runs resume without repeating side effects.

After approval gate, checkpoint stores pending payment until human approves, then continues.

**episodic memory**

Episodic memory stores past run trajectories—what was tried, what failed—for future reference within or across sessions.

Remembering last week's failed migration path prevents repeating the same broken sequence.

**recovery**

Recovery restores consistent state after crashes, tool failures, or partial commits. It requires durable checkpoints and compensating actions.

After payment timeout, recovery verifies ledger state before retry or refund.

**compensation**

Compensation undo or offsets partial effects when later steps fail—Saga pattern for agents. Without it, retries duplicate charges or records.

Failed booking after charge triggers automatic refund compensation transaction.

**idempotency**

Idempotent tools produce the same effect when called repeatedly with the same idempotency key. Agents retry safely only when tools support this.

create_ticket with idempotency key 'abc' must not spawn duplicate tickets on retry.

#### Worked example

**Situation:** Onboarding pauses overnight for manager approval; the agent must resume without recreating accounts or double-charging hardware orders.

**Baseline:** Store only chat transcript—restart loses progress and repeats writes.

**Application:** Persist checkpoints after each idempotent-safe step, durable episodic log, compensation actions for partial failures, resume from last committed checkpoint.

**Test cases:** (1) Normal: resume after clean pause. (2) Boundary: crash after non-idempotent step before checkpoint. (3) Adversarial: duplicate resume messages from two workers.

**Measurement:** Duplicate side-effect count, time-to-resume, checkpoint integrity checks.

**Design question:** Which steps require idempotency keys before they can be checkpointed safely?

**Predict before running.** Continuity requires durable state and recoverable effects, not merely longer context. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 8.3: Agent Memory and Recovery"""

checkpoints = []
state = {"step": "assign_laptop", "order_id": None}
def save(state):
    checkpoints.append(dict(state))
def resume():
    return checkpoints[-1] if checkpoints else None
save({"step": "create_account", "done": True})
state = resume()
print("resume_at:", state)


#### Interpret and apply

Persist and resume an interrupted multi-step run.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why is continuity more than longer context?
2. How does compensation differ from retry?
3. What recovery baseline relies on transcript only?

<details><summary>Check your reasoning</summary>

Q1: Durable state survives restarts; context windows do not. Q2: Compensation undoes partial effects; retry may duplicate them. Q3: Re-prompt model with chat history only.

</details>


### Lecture 04 · Agent Patterns

Apply planner–executor, supervisor–worker, reviewer, evaluator–optimizer, routing, and human-approval patterns.

#### A review step protects the moment a draft becomes a commitment

A general-purpose planner suggests a shift swap. It is plausible but violates a commitment to a volunteer made off-platform.

Use a draft–review–publish workflow and make the evidence available at the approval step. The person reviewing the proposal needs to see affected shifts and who will be notified.

#### Read the mechanism

**planner-executor**

Planner–executor splits strategic planning from tactical execution, often with different models or prompts. Plans can be validated before expensive actions.

Planner outputs step graph; executor calls tools one step at a time with verification.

**supervisor-worker**

Supervisor–worker assigns subtasks to workers and integrates results, adding coordination overhead for parallelizable work.

Supervisor delegates research subtopics to three workers, then merges citations.

**reviewer**

Reviewer pattern inserts a critique pass before delivery or irreversible actions. Reviewers should use different prompts or models than generators.

Draft email reviewed for PII leakage before send tool invocation.

**routing**

Routing directs requests to models, tools, or strategies by task type, risk, or budget. Routers encode product policy about cheap versus capable paths.

Simple FAQs route to small model; compliance questions route to audited large model.

**approval gates**

Approval gates pause execution until authorized humans confirm high-impact actions. They convert autonomy into supervised autonomy.

Production deploy agent waits for manager click before kubectl apply.

#### Worked example

**Situation:** Team debates planner–executor vs supervisor–worker patterns for onboarding with compliance review.

**Baseline:** Single monolithic agent prompt handling planning and execution.

**Application:** Implement two patterns: planner–executor with separate plans, and supervisor routing subtasks; measure coordination overhead, latency, and failure isolation.

**Test cases:** (1) Normal: linear subtasks. (2) Boundary: reviewer rejects one subtask. (3) Adversarial: worker agent returns plausible but unauthorized action.

**Measurement:** End-to-end latency, inter-agent messages, defect rate vs monolithic agent.

**Design question:** When does supervisor overhead exceed its fault-isolation benefit?

**Predict before running.** Patterns trade flexibility for additional state, calls, latency, and failure surfaces. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 8.4: Agent Patterns"""

patterns = {
    "monolith": {"calls": 1, "latency": 1.0},
    "planner_executor": {"calls": 3, "latency": 1.6},
    "supervisor_worker": {"calls": 5, "latency": 2.1},
}
task_risk = "high"
choice = "supervisor_worker" if task_risk == "high" else "planner_executor"
print({"pattern": choice, **patterns[choice]})


#### Interpret and apply

Implement two patterns and measure coordination overhead.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. What do agent patterns trade for flexibility?
2. When is a reviewer agent worth the extra call?
3. What baseline uses one prompt for everything?

<details><summary>Check your reasoning</summary>

Q1: Extra state, calls, latency, failure surfaces. Q2: High-risk actions needing independent gate. Q3: Single agent with combined plan+act instructions.

</details>


**Section checkpoint.** Record what must survive a restart so an approved action is handled once.


## Coordination and operations


### Lecture 05 · Multi-Agent Systems

Study delegation, role boundaries, communication, shared state, consensus, conflict, security, and why many tasks do not need multiple agents.

#### Two specialists disagree

One agent optimizes staffing and another minimizes changes to volunteers’ schedules. Both proposals pass their own tests.

Define a shared outcome and an explicit conflict-resolution rule. Let a reviewer see the trade-off instead of adding more agents. Track who made the final decision and which constraints they accepted.

#### Read the mechanism

**delegation**

Delegation assigns subtasks to specialized agents or tools with scoped permissions. Poor delegation boundaries cause duplicated work or authority gaps.

Legal sub-agent handles contract clauses; main agent cannot invoke legal tools directly.

**coordination**

Coordination synchronizes multiple agents—shared queues, locks, message passing—to avoid conflicting actions. It adds latency and failure modes.

Two workers must not edit the same document; lease coordinates exclusive access.

**shared state**

Shared state stores variables visible to multiple agents—task boards, evidence pools. Consistency requires versioning or transactional updates.

Research evidence store accumulates URLs all workers cite; stale entries need TTL.

**consensus**

Consensus protocols align multiple agents on a decision before action—voting, debate, or judge model. Useful when single-agent judgment is unreliable.

Three agents vote on classification before automated ticket routing proceeds.

**role isolation**

Role isolation restricts each agent to tools and data matching its role, limiting blast radius of compromise or error.

Billing agent cannot access HR records even if prompt requests it.

#### Worked example

**Situation:** Engineering proposes five specialized agents for onboarding; operations struggles with coordination failures.

**Baseline:** Five agents with shared scratchpad and no role isolation—conflicting writes.

**Application:** Split research-only subtask across workers with role boundaries, shared read-only evidence store, supervisor merge; compare to single agent with parallel tool calls.

**Test cases:** (1) Normal: parallel document fetches. (2) Boundary: two workers propose conflicting access levels. (3) Adversarial: compromised worker poisons shared state.

**Measurement:** Conflict incidents, total tokens, task success vs single-agent parallel tools.

**Design question:** When do parallel tools inside one agent suffice instead of multiple agents?

**Predict before running.** More agents increase organizational complexity faster than raw capability. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 8.5: Multi-Agent Systems"""

CHAPTER = "8.5"
print("chapter hook:", CHAPTER)
workers = {"A": "fetch HR policy", "B": "fetch IT policy"}
shared = []
for w, task in workers.items():
    shared.append({"worker": w, "result": f"evidence from {task}"})
conflicts = len({r["result"][:10] for r in shared}) < len(shared)
print({"evidence": shared, "conflict": conflicts})
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Split a research task across workers and compare with one-agent parallel tools.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why do more agents increase organizational complexity?
2. How does shared mutable state create security risk?
3. What simpler baseline parallelizes tool calls?

<details><summary>Check your reasoning</summary>

Q1: Delegation, consensus, and messaging multiply failure modes. Q2: One agent can overwrite another's conclusions. Q3: Single agent issuing parallel read-only tool calls.

</details>


### Lecture 06 · Operating Long-Running Agents

Design durable orchestration, queues, scheduling, leases, approvals, monitoring, incident response, and safe cancellation.

#### Late results are still real results

A calendar operation times out and returns success after the worker is ready to retry. Volunteers are already seeing a new shift.

Use a stable operation identifier and inspect the external result before retrying an uncertain effect. Record whether it was completed, pending, or requires reconciliation so the operator knows what to do after restart.

#### Read the mechanism

**durable execution**

Durable execution persists workflow state across process restarts and deploys—Temporal, Step Functions patterns. Long agents need this, not in-memory loops alone.

Day-long onboarding workflow survives server restart and resumes at last checkpoint.

**queues**

Queues decouple agent work submission from processing, smoothing load and enabling retries. Poison messages need dead-letter handling.

Approval tasks queue while humans respond; workers poll with backoff.

**leases**

Leases grant temporary exclusive ownership of a resource—document, ticket, shard—preventing duplicate processing. Expired leases must reclaim safely.

Worker holds 60s lease on ticket; another worker picks up only after lease expiry.

**human oversight**

Human oversight defines when and how people supervise agents—monitoring dashboards, escalation queues, kill switches. It scales only with clear triggers.

Escalate to human when confidence < 0.7 or spend > $1 on a single task.

**cancellation**

Cancellation stops in-flight agent work cleanly—revoke leases, abort tool calls, compensate partial effects. Users need cancel when plans change.

User cancels long research job; system stops tools and marks run cancelled, not failed.

#### Worked example

**Situation:** Onboarding agent runs up to 24 hours with human approvals; SRE needs SLOs and safe cancellation.

**Baseline:** Fire-and-forget background job with no lease or monitoring.

**Application:** Design durable orchestration with queue, worker leases, approval webhooks, heartbeat monitoring, cancel propagates to in-flight tools, runbook for stuck runs.

**Test cases:** (1) Normal: completes within SLO. (2) Boundary: approval waits 12 hours. (3) Adversarial: worker crash mid-lease without release.

**Measurement:** SLO adherence, stuck-run detection time, clean cancel success rate.

**Design question:** What lease duration balances slow approvals against fast failure detection?

**Predict before running.** Long-running agents are distributed systems with probabilistic decision components. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 8.6: Operating Long-Running Agents"""

CHAPTER = "8.6"
print("chapter hook:", CHAPTER)
SLO_HOURS = 24
lease_minutes = 30
elapsed = 12 * 60
renewals = elapsed // lease_minutes
print({"elapsed_min": elapsed, "lease_renewals": renewals, "within_slo": elapsed <= SLO_HOURS * 60})
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Create an SLO and runbook for a day-long agent workflow.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why are long-running agents distributed systems problems?
2. How do leases prevent duplicate workers?
3. What operations baseline lacks cancellation?

<details><summary>Check your reasoning</summary>

Q1: They need queues, timeouts, idempotency, human waits. Q2: Expired lease allows takeover; active lease blocks duplicate. Q3: Background thread with no orchestrator.

</details>


**Section checkpoint.** Choose a stopping rule and specify what evidence an operator receives on failure.


## Course workshop

Trace the example from planning to its approval pause, then resume after approval and verify that it reaches completion.

Run the reference implementation, then run the checks below it.


In [ ]:
"""Bounded agent state machine with checkpoint, approval, and termination."""
from dataclasses import dataclass, asdict
import json


@dataclass
class State:
    step: str = "plan"
    attempts: int = 0
    approved: bool = False
    done: bool = False


def transition(state):
    if state.attempts >= 5: return State("stopped", state.attempts, state.approved, True)
    if state.step == "plan": return State("draft", state.attempts + 1)
    if state.step == "draft": return State("approval", state.attempts + 1)
    if state.step == "approval" and not state.approved: return State("approval", state.attempts + 1)
    return State("complete", state.attempts + 1, state.approved, True)


state = State()
while not state.done:
    print(json.dumps(asdict(state)))
    if state.step == "approval": state.approved = True
    state = transition(state)
print(json.dumps(asdict(state)))


### Verify and extend

Run the assertions. Then change one assumption and add a check for the predicted behavior.


In [ ]:
state = State()
state = transition(transition(state))
assert state.step == "approval"
state = transition(state)
assert state.step == "approval"
state.approved = True
state = transition(state)
assert state.step == "complete" and state.done
print('Workshop checks passed')


## Course project

Build a durable multi-step agent with checkpoints, approval gates, evaluation, and observable termination.

Keep your baseline, results, and one failure case so someone else can review the decision you made.
